In [ ]:
import folium
from datetime import datetime
from sentinelhub import (
    SHConfig, SentinelHubStatistical, Geometry, 
    CRS, BBox, DataCollection, SentinelHubCatalog
)

# 1. CONFIGURATION
config = SHConfig()
config.sh_client_id = 'insert_client_id_here'  # Get from CDSE Identity Management
config.sh_client_secret = 'insert_secret_here'  # Get from CDSE Identity Management
config.sh_base_url = 'https://sh.dataspace.copernicus.eu' # CDSE Endpoint 
config.sh_token_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'

# Pilot Region: Homokhatsag (approximate coords)
HOMOKHATSAG_BBOX = BBox(bbox=[19.1, 46.3, 19.9, 46.9], crs=CRS.WGS84)

# 2. MOCK DATA (Hungarian Registry & Meter Telemetry)
# In production, these would be fetched from vizugy.hu or CSVs
farm_registry = [
    {
        "id": "HU_PARCEL_001",
        "name": "Kiskun Farm",
        "coords": [[[19.45, 46.55], [19.47, 46.55], [19.47, 46.57], [19.45, 46.57], [19.45, 46.55]]],
        "meter_reading": 450, # Irrigation volume in m3
    },
    {
        "id": "HU_PARCEL_002",
        "name": "Tisza Reach",
        "coords": [[[19.60, 46.70], [19.62, 46.70], [19.62, 46.72], [19.60, 46.72], [19.60, 46.70]]],
        "meter_reading": 1200, # Potential waste suspicion
    }
]

# 3. STATISTICAL API: NDVI (VEGETATION STRESS)
# This script runs in the cloud to calculate health without downloading rasters [cite: 2]
evalscript_ndvi = """
//VERSION=3
function setup() {
    return {
        input: [{
            bands: ["B04", "B08", "dataMask"]
        }],
        output: [
            {id: "ndvi", bands: 1},
            {id: "dataMask", bands: 1}
        ]
    };
}
function evaluatePixel(samples) {
    let ndvi = (samples.B08 - samples.B04) / (samples.B08 + samples.B04);
    return {
        ndvi: [ndvi],
        dataMask: [samples.dataMask]  // tells the Stats API which pixels are valid
    };
}
"""

# Define once, globally
SENTINEL2_CDSE = DataCollection.SENTINEL2_L2A.define_from(
    name="SENTINEL2_L2A_CDSE",
    service_url="https://sh.dataspace.copernicus.eu"
)

def get_farm_stats(polygon_coords):
    geometry = Geometry({'type': 'Polygon', 'coordinates': polygon_coords}, CRS.WGS84)

    stats_request = SentinelHubStatistical(
        aggregation=SentinelHubStatistical.aggregation(
            evalscript=evalscript_ndvi,
            time_interval=('2024-04-01', '2024-04-25'),
            aggregation_interval='P1D'
        ),
        input_data=[SentinelHubStatistical.input_data(
            data_collection=SENTINEL2_CDSE
        )],
        geometry=geometry,
        config=config
    )

    data = stats_request.get_data()[0]
    print("Raw response:", data)  # keep this temporarily to inspect

    # The top-level response is {"data": [...], "status": "OK"}
    # Each item in "data" is a daily interval
    intervals = data.get('data', [])
    
    valid_means = []
    for interval in intervals:
        outputs = interval.get('outputs', {})
        ndvi_bands = outputs.get('ndvi', {}).get('bands', {}).get('B0', {})
        stats = ndvi_bands.get('stats', {})
        
        # sampleCount=0 or noDataCount means no valid pixels that day
        sample_count = stats.get('sampleCount', 0)
        no_data_count = stats.get('noDataCount', 0)
        mean = stats.get('mean')
        
        if mean is not None and sample_count > no_data_count:
            valid_means.append(mean)
    
    if not valid_means:
        print("No valid NDVI data found for this polygon in the time range.")
        return 0.0
    
    return sum(valid_means) / len(valid_means)

# 4. SCORING MODEL (Rule-based)
def calculate_waste_score(meter_val, ndvi_val):
    """Simple logic: High water + Low health = Waste or Leakage."""
    # Estimated Need based on NDVI (very simplified)
    estimated_need = ndvi_val * 1000 
    waste_gap = meter_val - estimated_need
    
    # Normalize to 0-100 scale
    score = max(0, min(100, (waste_gap / 10)))
    return score

# 5. ASSEMBLE THE DASHBOARD
def build_map():
    m = folium.Map(location=[46.6, 19.5], zoom_start=9)
    
    for farm in farm_registry:
        # Step A: Get remote satellite stats [cite: 2]
        ndvi = get_farm_stats(farm['coords'])
        
        # Step B: Calculate risk 
        waste_score = calculate_waste_score(farm['meter_reading'], ndvi)
        
        # Color Logic: Red for high waste/risk
        color = 'red' if waste_score > 70 else 'orange' if waste_score > 40 else 'green'
        
        # Step C: Add to Folium Map
        folium.Polygon(
            locations=[(c[1], c[0]) for c in farm['coords'][0]],
            color=color,
            fill=True,
            fill_opacity=0.6,
            popup=(f"<b>{farm['name']}</b><br>"
                   f"Waste Score: {waste_score:.1f}<br>"
                   f"Meter: {farm['meter_reading']} m3<br>"
                   f"NDVI Health: {ndvi:.2f}")
        ).add_to(m)

    # Add Sentinel WMS Layer for visual context 
    folium.WmsTileLayer(
        url="https://sh.dataspace.copernicus.eu/ogc/wms/3d8eab64-0969-47f7-8314-b9142822a6f7",
        layers="TRUE_COLOR",
        name="Satellite View (Sentinel-2)",
        attr="Copernicus CDSE"
    ).add_to(m)
    
    folium.LayerControl().add_to(m)
    return m
# # RUN
# if __name__ == "__main__":
#     print("Fetching satellite metrics and ranking farms...")
#     inspector_map = build_map()
#     inspector_map.save("homokhatsag_risk_map.html")
#     print("Dashboard saved to: homokhatsag_risk_map.html")


In [5]:
inspector_map = build_map()
inspector_map.save("homokhatsag_risk_map.html")
print("Dashboard saved to: homokhatsag_risk_map.html")


Raw response: {'data': [{'interval': {'from': '2024-04-03T00:00:00Z', 'to': '2024-04-04T00:00:00Z'}, 'outputs': {'ndvi': {'bands': {'B0': {'stats': {'min': -0.022417154163122177, 'max': 0.9161439538002014, 'mean': 0.46497390799996313, 'stDev': 0.1476762601791722, 'sampleCount': 65536, 'noDataCount': 0}}}}}}, {'interval': {'from': '2024-04-08T00:00:00Z', 'to': '2024-04-09T00:00:00Z'}, 'outputs': {'ndvi': {'bands': {'B0': {'stats': {'min': 0.05919329449534416, 'max': 0.6169621348381042, 'mean': 0.3325788673316203, 'stDev': 0.07369774758464626, 'sampleCount': 65536, 'noDataCount': 0}}}}}}, {'interval': {'from': '2024-04-13T00:00:00Z', 'to': '2024-04-14T00:00:00Z'}, 'outputs': {'ndvi': {'bands': {'B0': {'stats': {'min': -0.007259528152644634, 'max': 0.8985945582389832, 'mean': 0.43554723082284535, 'stDev': 0.16555673042016622, 'sampleCount': 65536, 'noDataCount': 0}}}}}}, {'interval': {'from': '2024-04-18T00:00:00Z', 'to': '2024-04-19T00:00:00Z'}, 'outputs': {'ndvi': {'bands': {'B0': {'sta

In [7]:
display(inspector_map)

b'<html>\n<head>\n<meta http-equiv="Content-Type" content="text/html;charset=utf-8"/>\n<title>Error 500 Request failed.</title>\n</head>\n<body><h2>HTTP ERROR 500 Request failed.</h2>\n<table>\n<tr><th>URI:</th><td>/configuration/v1/wms/instances</td></tr>\n<tr><th>STATUS:</th><td>500</td></tr>\n<tr><th>MESSAGE:</th><td>Request failed.</td></tr>\n<tr><th>SERVLET:</th><td>org.glassfish.jersey.servlet.ServletContainer-7dddfc35</td></tr>\n</table>\n\n</body>\n</html>\n'
